In [1]:


import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from datetime import datetime
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
plt.rcParams['axes.unicode_minus'] = False    # 用来正常显示负号




In [2]:
train_path=r"D:\python专业版项目\Data science\competitions\titanic\data\raw\train.csv"
test_path=r"D:\python专业版项目\Data science\competitions\titanic\data\raw\test.csv"

train=pd.read_csv(train_path)
test=pd.read_csv(test_path)

In [3]:
#特征工程前，对训练集和测试集进行处理。注意采用不同的方式进行处理


train = train.copy()
test = test.copy()
# 复制数据，避免修改原始数据

# 1. 处理 Age：用中位数填充（年龄分布有偏，中位数更稳健）
median_age = train['Age'].median()
train['Age'] = train['Age'].fillna(median_age)
test['Age'] = test['Age'].fillna(median_age)

print(f"Age缺失值处理完成，填充值: {median_age}")
#这里年龄的偏度较小，所以采用中位数对缺失值进行填充，更加稳健
# 2. 处理 Embarked：只有2个缺失，用众数填充
most_common_embarked = train['Embarked'].mode()[0]
train['Embarked'] = train['Embarked'].fillna(most_common_embarked)
#对训练集进行用众数填充，这里是因为港口的缺失值较少，适合用众数进行填充

if test['Embarked'].isnull().sum() > 0:
    test['Embarked'] = test['Embarked'].fillna(most_common_embarked)

#对测试集进行填充，如果测试集存在缺失的话，避免后面因为缺失而报错
print(f"Embarked缺失值处理完成，填充值: {most_common_embarked}")

train['Has_Cabin'] = train['Cabin'].notna().astype(int)
test['Has_Cabin'] = test['Cabin'].notna().astype(int)

print("Has_Cabin特征创建完成，1=有船舱号，0=无")
print(train['Has_Cabin'].value_counts())

# 4. 可以删除原始Cabin列（后续不用）
train = train.drop('Cabin', axis=1)
test = test.drop('Cabin', axis=1)

Age缺失值处理完成，填充值: 28.0
Embarked缺失值处理完成，填充值: S
Has_Cabin特征创建完成，1=有船舱号，0=无
Has_Cabin
0    687
1    204
Name: count, dtype: int64


In [4]:
#————————————————————————
#开始进行特征工程


In [5]:
# 3.1 从姓名提取称呼（Title）
# 观察：不同称呼（Mr, Mrs, Miss, Master）可能反映年龄和社会地位
train['Title'] = train['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test['Title'] = test['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


print("\n训练集中的称呼分布：")
print(train['Title'].value_counts())

title_mapping = {
    "Mr": "Mr",
    "Miss": "Miss",
    "Mrs": "Mrs",
    "Master": "Master",
    "Dr": "Rare",
    "Rev": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Mlle": "Rare",
    "Countess": "Rare",
    "Ms": "Rare",
    "Lady": "Rare",
    "Jonkheer": "Rare",
    "Don": "Rare",
    "Dona": "Rare",
    "Mme": "Rare",
    "Capt": "Rare",
    "Sir": "Rare"
}

train['Title'] = train['Title'].map(title_mapping)
test['Title'] = test['Title'].map(title_mapping)
#创建一个字典，使用map函数。合并稀有称呼。合并后，特征更加简洁工整。利于模型训练

# 独热编码称呼
train = pd.get_dummies(train, columns=['Title'], prefix='Title')
test = pd.get_dummies(test, columns=['Title'], prefix='Title')
#独热编码，用于将Mr, Mrs, Miss, Master，Rare映射成一个向量，机器学习算法只能识别这些向量

print("\n称呼编码完成！")

# 3.2 年龄分组（AgeBin）
# 根据之前的观察，不同年龄段幸存率不同
bins = [0, 12, 18, 35, 60, 100]
labels = ['Child', 'Teen', 'Adult', 'MiddleAged', 'Elderly']
train['Agebin'] = pd.cut(train['Age'], bins=bins, labels=labels)
test['Agebin'] = pd.cut(test['Age'], bins=bins, labels=labels)


# 独热编码年龄组
train = pd.get_dummies(train, columns=['Agebin'], prefix='Age')
test = pd.get_dummies(test, columns=['Agebin'], prefix='Age')

print("年龄分组编码完成！")


# 3.3 票价分组（FareBin）
# 将票价分成几个档次


thresholds = [0, 20, 45, 100, 200, 550]
labelss = ['Poor', 'LowMiddle', 'Middle', 'UpperMiddle', 'Elite']

train['FareBin'] = pd.cut(train['Fare'], bins=thresholds, labels=labelss)
test['FareBin'] = pd.cut(test['Fare'], bins=thresholds, labels=labelss)

test.loc[test['Fare'] > train['Fare'].max(), 'Fare'] = train['Fare'].max()

# 独热编码票价组
train = pd.get_dummies(train, columns=['FareBin'], prefix='Fare')
test = pd.get_dummies(test, columns=['FareBin'], prefix='Fare')

print("票价分组编码完成！")

# 3.4 家庭特征组合

train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1

bins = [0, 1, 4, float('inf')]
labels = ['Alone', 'Small', 'Large']

train['FamilyType'] = pd.cut(train['FamilySize'], bins=bins, labels=labels)
test['FamilyType'] = pd.cut(test['FamilySize'], bins=bins, labels=labels)

# 独热编码
train = pd.get_dummies(train, columns=['FamilyType'], prefix='Family')
test = pd.get_dummies(test, columns=['FamilyType'], prefix='Family')

print("家庭类型特征创建完成！")

# 3.5 特征交叉（选做，可以先用简单的）
# 例如：性别+舱位 的组合
train['Sex_Pclass'] = train['Sex'].astype(str) + '_' + train['Pclass'].astype(str)
test['Sex_Pclass'] = test['Sex'].astype(str) + '_' + test['Pclass'].astype(str)


train = pd.get_dummies(train, columns=['Sex_Pclass'], prefix='SexPclass')
test = pd.get_dummies(test, columns=['Sex_Pclass'], prefix='SexPclass')

# print("特征交叉完成！")


train = pd.get_dummies(train, columns=['Embarked'], prefix='Embarked')
test = pd.get_dummies(test, columns=['Embarked'], prefix='Embarked')

print("\n 特征工程完成！")
print(f"当前训练集列数: {len(train.columns)}")
print(f"当前测试集列数: {len(test.columns)}")

#这里采用特征交叉的方式，生成新的特征，比单独看性别和舱位号更好



训练集中的称呼分布：
Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Major         2
Mlle          2
Col           2
Don           1
Mme           1
Ms            1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Name: count, dtype: int64

称呼编码完成！
年龄分组编码完成！
票价分组编码完成！
家庭类型特征创建完成！

✅ 特征工程完成！
当前训练集列数: 39
当前测试集列数: 38


<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:4: SyntaxWarning: invalid escape sequence '\.'
<>:3: SyntaxWarning: invalid escape sequence '\.'
<>:4: SyntaxWarning: invalid escape sequence '\.'
C:\Users\一氧化二氧\AppData\Local\Temp\ipykernel_43520\3030110216.py:3: SyntaxWarning: invalid escape sequence '\.'
  train['Title'] = train['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
C:\Users\一氧化二氧\AppData\Local\Temp\ipykernel_43520\3030110216.py:4: SyntaxWarning: invalid escape sequence '\.'
  test['Title'] = test['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


In [6]:
# 在保存之前添加这段调试代码
print("\n=== 数据类型检查 ===")
print("训练集每列的数据类型：")
print(train.dtypes)

print("\n哪些列是对象（文本）类型：")
object_cols = train.select_dtypes(include=['object']).columns.tolist()
print(object_cols)

if object_cols:
    print("\n⚠ 还有文本列没有编码！")
    print( object_cols)
else:
    print("所有列都已转换为数值！")


=== 数据类型检查 ===
训练集每列的数据类型：
PassengerId             int64
Survived                int64
Pclass                  int64
Name                      str
Sex                       str
Age                   float64
SibSp                   int64
Parch                   int64
Ticket                    str
Fare                  float64
Has_Cabin               int64
Title_Master             bool
Title_Miss               bool
Title_Mr                 bool
Title_Mrs                bool
Title_Rare               bool
Age_Child                bool
Age_Teen                 bool
Age_Adult                bool
Age_MiddleAged           bool
Age_Elderly              bool
Fare_Poor                bool
Fare_LowMiddle           bool
Fare_Middle              bool
Fare_UpperMiddle         bool
Fare_Elite               bool
FamilySize              int64
Family_Alone             bool
Family_Small             bool
Family_Large             bool
SexPclass_female_1       bool
SexPclass_female_2       bool
SexPclass_fe

C:\Users\一氧化二氧\AppData\Local\Temp\ipykernel_43520\2326969392.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = train.select_dtypes(include=['object']).columns.tolist()


In [7]:



# 1. 编码 Sex（用0/1编码更简单）
print("\n1. 编码 Sex 列...")
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})
print("\n3. 处理 Ticket 列...")
# 方案A：直接丢弃（因为太杂乱）
train = train.drop('Ticket', axis=1)
test = test.drop('Ticket', axis=1)




# 4. 处理 Name（姓名 - 我们已经提取了Title，可以丢弃）
print("\n4. 处理 Name 列...")
# 保留 Title 特征（已经创建），丢弃原始 Name
train = train.drop('Name', axis=1)


# 5. 最终检查
print("\n" + "=" * 50)
print("最终数据类型检查:")
print(train.dtypes.value_counts())
print("\n剩余列:")
print(train.columns.tolist())

# 确认没有文本列了
object_cols = train.select_dtypes(include=['object']).columns.tolist()


开始编码文本列...

1. 编码 Sex 列...
✅ Sex 编码完成 (male=0, female=1)

3. 处理 Ticket 列...
✅ Ticket 列已丢弃（太杂乱，不易提取有用信息）

4. 处理 Name 列...
✅ Name 列已丢弃（已提取Title特征）

最终数据类型检查:
bool       27
int64       8
float64     2
Name: count, dtype: int64

剩余列:
['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Has_Cabin', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Age_Child', 'Age_Teen', 'Age_Adult', 'Age_MiddleAged', 'Age_Elderly', 'Fare_Poor', 'Fare_LowMiddle', 'Fare_Middle', 'Fare_UpperMiddle', 'Fare_Elite', 'FamilySize', 'Family_Alone', 'Family_Small', 'Family_Large', 'SexPclass_female_1', 'SexPclass_female_2', 'SexPclass_female_3', 'SexPclass_male_1', 'SexPclass_male_2', 'SexPclass_male_3', 'Embarked_C', 'Embarked_Q', 'Embarked_S']

✅ 所有文本列都已处理完成！


In [8]:
import os
import shutil

# 1. 定义要删除的路径（根据你的项目结构）
# ① notebook下的data文件夹
notebook_data_dir = r'D:\python专业版项目\Data science\competitions\titanic\notebooks\data'
# ② titanic/data/processed里的旧数据文件
processed_dir = r'D:\python专业版项目\Data science\competitions\titanic\data\processed'
old_files = [
    os.path.join(processed_dir, 'X_train.csv'),
    os.path.join(processed_dir, 'X_test.csv'),
    os.path.join(processed_dir, 'y_train.csv')
]

# 2. 删除notebook下的data文件夹（如果存在）
if os.path.exists(notebook_data_dir):
    shutil.rmtree(notebook_data_dir)
    print(f" {notebook_data_dir}")
else:
    print(f"{notebook_data_dir} 不存在，无需删除")

# 3. 删除processed里的旧数据文件
for file in old_files:
    if os.path.exists(file):
        os.remove(file)
        print(f"已删除旧文件: {file}")
    else:
        print(f"{file} 不存在，无需删除")

print("\n 旧文件清理完成！")

⚠️ D:\python专业版项目\Data science\competitions\titanic\notebooks\data 不存在，无需删除
✅ 已删除旧文件: D:\python专业版项目\Data science\competitions\titanic\data\processed\X_train.csv
✅ 已删除旧文件: D:\python专业版项目\Data science\competitions\titanic\data\processed\X_test.csv
✅ 已删除旧文件: D:\python专业版项目\Data science\competitions\titanic\data\processed\y_train.csv

✅ 旧文件清理完成！


In [9]:

import os


processed_dir = r'D:\python专业版项目\Data science\competitions\titanic\data\processed'
# 确保目录存在（不存在则创建，存在则不报错）
os.makedirs(processed_dir, exist_ok=True)

# 2. 选择用于建模的特征（排除不需要的列）
exclude_cols = ['PassengerId', 'Name', 'Ticket', 'Survived']
feature_cols = [col for col in train.columns if col not in exclude_cols]

# 3. 准备训练/测试数据
X_train = train[feature_cols]
y_train = train['Survived']
X_test = test[feature_cols]

# 确保测试集和训练集列一致
X_test = X_test[X_train.columns]

X_train.to_csv(
    os.path.join(processed_dir, 'X_train.csv'),
    index=False,
    mode='w'  # 强制覆盖模式
)
X_test.to_csv(
    os.path.join(processed_dir, 'X_test.csv'),
    index=False,
    mode='w'
)
y_train.to_csv(
    os.path.join(processed_dir, 'y_train.csv'),
    index=False,
    mode='w'
)



# 查看最终使用的特征
print(X_train.columns.tolist())


✅ 数据已保存到指定路径：
📂 保存目录: D:\python专业版项目\Data science\competitions\titanic\data\processed
✅ X_train.csv (特征): (891, 35)
✅ y_train.csv (标签): (891,)
✅ X_test.csv (测试特征): (418, 35)

📋 最终使用的特征列表：
['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Has_Cabin', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Age_Child', 'Age_Teen', 'Age_Adult', 'Age_MiddleAged', 'Age_Elderly', 'Fare_Poor', 'Fare_LowMiddle', 'Fare_Middle', 'Fare_UpperMiddle', 'Fare_Elite', 'FamilySize', 'Family_Alone', 'Family_Small', 'Family_Large', 'SexPclass_female_1', 'SexPclass_female_2', 'SexPclass_female_3', 'SexPclass_male_1', 'SexPclass_male_2', 'SexPclass_male_3', 'Embarked_C', 'Embarked_Q', 'Embarked_S']


In [10]:
# ============================================
# 6. 重新训练模型并生成提交文件
# ============================================
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from datetime import datetime


param_grid = {'C': [0.5, 0.6, 0.7, 0.8, 0.9,1,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9]}

# 创建逻辑回归模型（注意：需先处理数据泄露！）
lr = LogisticRegression(max_iter=1000, random_state=42)

# 5折交叉验证网格搜索
grid = GridSearchCV(lr, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print(f"最佳C值: {grid.best_params_['C']}")
print(f"最佳交叉验证准确率: {grid.best_score_:.4f}")

best_model = grid.best_estimator_

# 2. 检查训练集准确率
train_pred = best_model.predict(X_train)
from sklearn.metrics import accuracy_score
train_acc = accuracy_score(y_train, train_pred)
print(f"训练集准确率: {train_acc:.4f}")


print(f"目前测试集中的缺失值：{X_test.isnull().sum()}")

X_test.fillna(X_train.mean(), inplace=True)

print(f"目前测试集中的缺失值：{X_test.isnull().sum()}")
# 3. 预测测试集
test_pred = best_model.predict(X_test)

# 4. 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': test_pred
})

# 5. 保存文件
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
submission_path = rf'D:\python专业版项目\Data science\competitions\titanic\submissions\submission_{timestamp}.csv'
submission.to_csv(submission_path, index=False)

print(f"\n提交文件已生成: {submission_path}")
print("\n文件前5行:")
print(submission.head())

# 6. 查看预测分布
print(f"\n预测结果分布:")
print(submission['Survived'].value_counts())
print(f"幸存比例: {submission['Survived'].mean():.2%}")



训练最终模型并生成提交文件
最佳C值: 0.9
最佳交叉验证准确率: 0.8249
训练集准确率: 0.8429
目前测试集中的缺失值：Pclass                0
Sex                   0
Age                   0
SibSp                 0
Parch                 0
Fare                  1
Has_Cabin             0
Title_Master          0
Title_Miss            0
Title_Mr              0
Title_Mrs             0
Title_Rare            0
Age_Child             0
Age_Teen              0
Age_Adult             0
Age_MiddleAged        0
Age_Elderly           0
Fare_Poor             0
Fare_LowMiddle        0
Fare_Middle           0
Fare_UpperMiddle      0
Fare_Elite            0
FamilySize            0
Family_Alone          0
Family_Small          0
Family_Large          0
SexPclass_female_1    0
SexPclass_female_2    0
SexPclass_female_3    0
SexPclass_male_1      0
SexPclass_male_2      0
SexPclass_male_3      0
Embarked_C            0
Embarked_Q            0
Embarked_S            0
dtype: int64
目前测试集中的缺失值：Pclass                0
Sex                   0
Age                 

'\n逻辑回归（Logistic Regression）是一种用于二分类问题的线性模型。尽管名字里有“回归”，但它实际上是分类算法。它的核心思想是：用一个线性函数拟合特征与目标的关系，然后通过Sigmoid函数将输出压缩到0~1之间，解释为属于某一类的概率\n简单快速：训练速度快，适合作为基线模型，快速验证数据处理和特征工程是否正确。\n\n可解释性强：可以查看每个特征的系数，直接判断哪些特征对幸存概率影响大（正系数增加幸存概率，负系数减少）。\n\n概率输出：可以直接输出预测概率，便于后续分析和集成。\n\n对线性关系有效：Titanic中许多特征（如性别、舱位）与幸存存在明显的线性关系，逻辑回归能很好地捕捉。\n\n正则化：通过参数C可以控制模型复杂度，防止过拟合。\n'

In [ ]:
#best_model = LogisticRegression(max_iter=1000, random_state=42, C=1.0)------0.77511